# ==============================================================================
# 🚲 URBAN MOBILITY LOGISTICS: BIKE SHARING DEMAND VIA RANDOM FOREST REGRESSOR
# ==============================================================================
### Core Principles:
Forecasting daily bike rental demand based on weather conditions (temperature, humidity, windspeed), season, and calendar holidays.
Random Forest models complex interaction effects between temperature and working days.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Urban Mobility Environment ready.")


✅ STEP 1: Urban Mobility Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle Bike Sharing Dataset (730 daily records).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/bike_sharing/day.csv'
df = pd.read_csv(data_path)

# Drop non-predictive metadata
df = df.drop(columns=['instant', 'dteday', 'casual', 'registered'])

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} daily records, {df.shape[1]} attributes")
print(f"Target Demand Summary:\n{df['cnt'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 730 daily records, 12 attributes
Target Demand Summary:
count     730.00
mean     4508.01
std      1936.01
min        22.00
25%      3169.75
50%      4548.50
75%      5966.00
max      8714.00
Name: cnt, dtype: float64


,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,0,1,0,6,0,2,14.110847,18.18125,80.5833,10.749882,985
1,1,0,1,0,0,0,2,14.902598,17.68695,69.6087,16.652113,801
2,1,0,1,0,1,1,1,8.050924,9.47025,43.7273,16.636703,1349


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `cnt` (total daily rentals).


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['cnt'])
y = df['cnt']

print(f"Missing attributes: {X.isnull().sum().sum()}")


Missing attributes: 0


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Days: {X_train.shape[0]}")
print(f"Test Days : {X_test.shape[0]}")


Train Days: 584
Test Days : 146


## ⚙️ STEP 7: SCALE-INVARIANT PREPROCESSOR
All weather, seasonal, and calendar features are scale-invariant numbers.


In [5]:
# ==============================================================================
# ⚙️ STEP 7: TREE-OPTIMIZED UNIVERSAL PREPROCESSOR (SCALE-INVARIANT)
# ==============================================================================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

def build_tree_preprocessor(X):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    transformers = []
    if len(num_cols) > 0:
        transformers.append(('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols))
    if len(cat_cols) > 0:
        transformers.append(('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
        ]), cat_cols))
    return ColumnTransformer(transformers=transformers, remainder='drop') if len(transformers) > 0 else 'passthrough'

preprocessor = build_tree_preprocessor(X_train)
print(f"✅ STEP 7: Tree-Optimized Universal Preprocessor Ready.")

✅ STEP 7: Tree-Optimized Universal Preprocessor Ready.


## ⚠️ STEP 8: UNCONSTRAINED BASELINE FIT (OVERFITTING PROOF)
Evaluating baseline Random Forest on bike fleet demand.


In [6]:
# STEP 8: UNCONSTRAINED BASELINE FIT (OVERFITTING PROOF)
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)

r2_base = r2_score(y_test, pipe_rf.predict(X_test))
print(f"Baseline Random Forest R² : {r2_base*100:.2f}%")
print(f"Baseline MAE              : {mean_absolute_error(y_test, pipe_rf.predict(X_test)):.2f} bikes")

Baseline Random Forest R² : 91.14%
Baseline MAE              : 414.77 bikes


## 🎯 STEP 9: HYPERPARAMETER TUNING VIA 5-FOLD CV
Optimizing `max_depth` and `min_samples_leaf`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'rf__n_estimators': [100, 150, 200],
    'rf__max_depth': [6, 10, 14],
    'rf__min_samples_leaf': [1, 2, 4]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])
grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²   : {grid.best_score_*100:.2f}%")

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/python/03_Custom_ad

🏆 Best Hyperparameters: {'rf__max_depth': 10, 'rf__min_samples_leaf': 1, 'rf__n_estimators': 200}
🎯 Best 5-Fold CV R²   : 86.56%


## ⚔️ STEP 10: ALGORITHM DUEL (Random Forest vs Single Tree)
Benchmarking ensemble gain against a single decision tree.


In [8]:
# STEP 10: BENCHMARK
from sklearn.tree import DecisionTreeRegressor

pipe_single = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeRegressor(random_state=42))
])
pipe_single.fit(X_train, y_train)

r2_tree = r2_score(y_test, pipe_single.predict(X_test))
r2_rf = r2_score(y_test, best_model.predict(X_test))

print("="*65)
print(f"Single Decision Tree -> R²: {r2_tree*100:.2f}% | MAE: {mean_absolute_error(y_test, pipe_single.predict(X_test)):.2f} bikes")
print(f"Random Forest (Tuned)-> R²: {r2_rf*100:.2f}% | MAE: {mean_absolute_error(y_test, best_model.predict(X_test)):.2f} bikes")
print("="*65)

Single Decision Tree -> R²: 83.55% | MAE: 570.08 bikes
Random Forest (Tuned)-> R²: 91.14% | MAE: 414.28 bikes


## 📊 STEP 11: RESIDUAL ANALYSIS & FLEET METRICS
Reviewing fleet dispatch metrics.


In [9]:
# STEP 11: FEATURE IMPORTANCE & ERROR DIAGNOSTICS
y_pred_rf = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"📋 URBAN FLEET DISPATCH METRICS:")
print(f"   Mean Absolute Error (MAE) : {mae:.2f} bikes")
print(f"   Root Mean Squared Error   : {rmse:.2f} bikes")
print(f"   R² Score                  : {r2_score(y_test, y_pred_rf)*100:.2f}%")


📋 URBAN FLEET DISPATCH METRICS:
   Mean Absolute Error (MAE) : 414.28 bikes
   Root Mean Squared Error   : 550.80 bikes
   R² Score                  : 91.14%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & DISPATCH SMOKE TEST
Deploying pipeline and validating fleet reallocation latency.


In [10]:
# STEP 12 & 13: PRODUCTION SERIALIZATION & LATENCY SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/bike_sharing_random_forest_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_day = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_bikes = loaded_pipe.predict(sample_day)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🚲 LIVE URBAN FLEET DISPATCH AUDIT:")
print(f"   Forecast Daily Demand: 🚲 {int(pred_bikes):,} bikes")
print(f"   Actual Rentals       : {int(y_test.iloc[0]):,} bikes")
print(f"   Dispatch Latency     : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/bike_sharing_random_forest_pipeline.joblib (1,491,382 bytes)



🚲 LIVE URBAN FLEET DISPATCH AUDIT:
   Forecast Daily Demand: 🚲 6,208 bikes
   Actual Rentals       : 7,460 bikes
   Dispatch Latency     : 43.349 ms (SLA < 2.0ms: CHECK)
